In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# ============================================================
# Rule-based Synthetic Event Log Generator
# Method : First-order Markov Chain
# Input  : ./../MIMICEL_data/mimicel_train.csv
# Output : ./results/rule_based_synthetic_data_seedXX.csv
# ============================================================

# -----------------------------
# Path
# -----------------------------
DATA_DIR = Path("./../MIMICEL_data")
TRAIN_PATH = DATA_DIR / "mimicel_train.csv"

RESULT_DIR = Path("./results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Columns
# -----------------------------
CASE_COL = "stay_id"
ACT_COL = "activity"
TIME_COL = "timestamps"

# -----------------------------
# Generation config
# -----------------------------
SEEDS = [41, 42, 43, 44, 45]

N_SYN_CASES = None          # None이면 train case 수와 동일하게 생성
MAX_TRACE_LEN = 100         # 무한 루프 방지
MIN_TRACE_LEN = 3           # 너무 짧은 trace 방지

START = "__START__"
END = "__END__"

# ============================================================
# 1. Load train data
# ============================================================

df = pd.read_csv(TRAIN_PATH)

required_cols = [CASE_COL, ACT_COL, TIME_COL]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(
        f"Missing columns: {missing_cols}\n"
        f"Available columns: {df.columns.tolist()}"
    )

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.dropna(subset=[CASE_COL, ACT_COL, TIME_COL]).copy()
df = df.sort_values([CASE_COL, TIME_COL]).reset_index(drop=True)

if N_SYN_CASES is None:
    N_SYN_CASES = df[CASE_COL].nunique()

print(f"[LOAD] events={len(df):,}, cases={df[CASE_COL].nunique():,}")
print(f"[INFO] synthetic cases per seed={N_SYN_CASES:,}")

# ============================================================
# 2. Build real traces
# ============================================================

traces = (
    df.groupby(CASE_COL)[ACT_COL]
    .apply(lambda x: x.tolist())
    .tolist()
)

train_variants = set(tuple(t) for t in traces)
trace_lengths = np.array([len(t) for t in traces])

print(f"[TRACE] unique variants={len(train_variants):,}")
print(f"[TRACE] mean length={trace_lengths.mean():.2f}, max length={trace_lengths.max()}")

# ============================================================
# 3. Build first-order Markov transition probability
# ============================================================

transition_counts = defaultdict(Counter)

for trace in traces:
    if len(trace) == 0:
        continue

    transition_counts[START][trace[0]] += 1

    for a, b in zip(trace[:-1], trace[1:]):
        transition_counts[a][b] += 1

    transition_counts[trace[-1]][END] += 1

transition_probs = {}

for src, counter in transition_counts.items():
    next_acts = list(counter.keys())
    counts = np.array(list(counter.values()), dtype=float)
    probs = counts / counts.sum()

    transition_probs[src] = {
        "next": next_acts,
        "prob": probs
    }

print(f"[MARKOV] states={len(transition_probs):,}")

# ============================================================
# 4. Build time-delta distribution by transition
# ============================================================

df["_next_activity"] = df.groupby(CASE_COL)[ACT_COL].shift(-1)
df["_next_time"] = df.groupby(CASE_COL)[TIME_COL].shift(-1)

trans_df = df.dropna(subset=["_next_activity", "_next_time"]).copy()

trans_df["_delta_seconds"] = (
    trans_df["_next_time"] - trans_df[TIME_COL]
).dt.total_seconds()

trans_df = trans_df[trans_df["_delta_seconds"] >= 0].copy()

transition_to_deltas = {
    (a, b): g["_delta_seconds"].values
    for (a, b), g in trans_df.groupby([ACT_COL, "_next_activity"])
}

global_deltas = trans_df["_delta_seconds"].values
global_deltas = global_deltas[global_deltas >= 0]

start_times = df.groupby(CASE_COL)[TIME_COL].min().values

print(f"[TIME] transition delta types={len(transition_to_deltas):,}")

# ============================================================
# 5. Build attribute sampling pools
# ============================================================

activity_to_event_pool = {
    act: g.copy()
    for act, g in df.groupby(ACT_COL)
}

case_first_rows = (
    df.sort_values([CASE_COL, TIME_COL])
    .groupby(CASE_COL)
    .first()
    .reset_index()
)

output_columns = [
    c for c in df.columns
    if c not in ["_next_activity", "_next_time", "_delta_seconds"]
]

# ============================================================
# 6. Generate synthetic logs for each seed
# ============================================================

summary_rows = []

for SEED in SEEDS:

    rng = np.random.default_rng(SEED)

    OUT_PATH = RESULT_DIR / f"rule_based_synthetic_data_seed{SEED}.csv"

    # --------------------------------------------------------
    # Helper functions
    # --------------------------------------------------------

    def sample_next_activity(current_act):
        if current_act not in transition_probs:
            return END

        candidates = transition_probs[current_act]["next"]
        probs = transition_probs[current_act]["prob"]

        return rng.choice(candidates, p=probs)


    def generate_trace():
        trace = []
        current = START

        for _ in range(MAX_TRACE_LEN):
            nxt = sample_next_activity(current)

            if nxt == END:
                if len(trace) >= MIN_TRACE_LEN:
                    break

                candidates = transition_probs[current]["next"]
                probs = transition_probs[current]["prob"]

                non_end = [
                    (a, p)
                    for a, p in zip(candidates, probs)
                    if a != END
                ]

                if len(non_end) == 0:
                    break

                acts, ps = zip(*non_end)
                ps = np.array(ps, dtype=float)
                ps = ps / ps.sum()

                nxt = rng.choice(acts, p=ps)

            trace.append(nxt)
            current = nxt

        return trace


    def sample_delta_seconds(prev_act, next_act):
        arr = transition_to_deltas.get((prev_act, next_act), None)

        if arr is None or len(arr) == 0:
            arr = global_deltas

        if arr is None or len(arr) == 0:
            return 1.0

        delta = float(rng.choice(arr))

        if delta <= 0:
            delta = 1.0

        return delta


    def sample_event_row(activity):
        pool = activity_to_event_pool.get(activity, None)

        if pool is None or len(pool) == 0:
            return df.iloc[rng.integers(0, len(df))]

        return pool.iloc[rng.integers(0, len(pool))]


    def sample_case_template():
        return case_first_rows.iloc[rng.integers(0, len(case_first_rows))]


    def make_new_case_id(i):
        return f"syn_{SEED}_{i:07d}"

    # --------------------------------------------------------
    # Main generation
    # --------------------------------------------------------

    synthetic_rows = []

    for i in range(1, N_SYN_CASES + 1):

        new_case_id = make_new_case_id(i)
        trace = generate_trace()

        if len(trace) == 0:
            continue

        case_template = sample_case_template()
        current_time = pd.Timestamp(rng.choice(start_times))

        for pos, act in enumerate(trace):

            event_template = sample_event_row(act)
            row = {}

            for col in output_columns:

                if col == CASE_COL:
                    row[col] = new_case_id

                elif col == ACT_COL:
                    row[col] = act

                elif col == TIME_COL:
                    row[col] = current_time

                else:
                    if col in case_template.index and pd.notna(case_template[col]):
                        row[col] = case_template[col]
                    elif col in event_template.index:
                        row[col] = event_template[col]
                    else:
                        row[col] = np.nan

            synthetic_rows.append(row)

            if pos < len(trace) - 1:
                next_act = trace[pos + 1]
                delta_sec = sample_delta_seconds(act, next_act)
                current_time = current_time + pd.to_timedelta(delta_sec, unit="s")

    syn_df = pd.DataFrame(synthetic_rows)
    syn_df = syn_df[output_columns]

    syn_df[TIME_COL] = (
        pd.to_datetime(syn_df[TIME_COL])
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

    syn_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------

    syn_variants = (
        syn_df.groupby(CASE_COL)[ACT_COL]
        .apply(lambda x: tuple(x.tolist()))
    )

    new_variant_ratio = (~syn_variants.isin(train_variants)).mean()

    mean_trace_length = syn_df.groupby(CASE_COL).size().mean()
    max_trace_length = syn_df.groupby(CASE_COL).size().max()

    summary_rows.append({
        "seed": SEED,
        "output_file": OUT_PATH.name,
        "synthetic_cases": syn_df[CASE_COL].nunique(),
        "synthetic_events": len(syn_df),
        "synthetic_variants": syn_variants.nunique(),
        "new_variant_ratio": new_variant_ratio,
        "mean_trace_length": mean_trace_length,
        "max_trace_length": max_trace_length
    })

    print(
        f"[SEED {SEED}] "
        f"saved={OUT_PATH}, "
        f"cases={syn_df[CASE_COL].nunique():,}, "
        f"events={len(syn_df):,}, "
        f"variants={syn_variants.nunique():,}, "
        f"new_variant_ratio={new_variant_ratio:.4f}"
    )

# ============================================================
# 7. Save summary
# ============================================================

summary_df = pd.DataFrame(summary_rows)

SUMMARY_PATH = RESULT_DIR / "rule_based_synthetic_summary.csv"
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"[SAVE SUMMARY] {SUMMARY_PATH}")
print("[DONE]")

[LOAD] events=16,176, cases=899
[INFO] synthetic cases per seed=899
[TRACE] unique variants=785
[TRACE] mean length=17.99, max length=100
[MARKOV] states=7
[TIME] transition delta types=24
[SEED 41] saved=results\rule_based_synthetic_data_seed41.csv, cases=899, events=16,564, variants=712, new_variant_ratio=0.8009
[SEED 42] saved=results\rule_based_synthetic_data_seed42.csv, cases=899, events=16,988, variants=685, new_variant_ratio=0.7875
[SEED 43] saved=results\rule_based_synthetic_data_seed43.csv, cases=899, events=15,814, variants=688, new_variant_ratio=0.7798
[SEED 44] saved=results\rule_based_synthetic_data_seed44.csv, cases=899, events=16,326, variants=694, new_variant_ratio=0.7887
[SEED 45] saved=results\rule_based_synthetic_data_seed45.csv, cases=899, events=15,703, variants=701, new_variant_ratio=0.8042
[SAVE SUMMARY] results\rule_based_synthetic_summary.csv
[DONE]


: 